# 00 — Smoke Test

**Purpose:** verify that everything in our environment works before we touch any real data.

Run every cell top-to-bottom. If all green checkmarks ✅ appear at the end, Phase 1 is complete.

**What this notebook does:**
1. Verifies GPU is available
2. Mounts Google Drive and creates folder structure
3. Installs pinned dependencies
4. Loads Wav2Vec 2.0 from HuggingFace
5. Runs one forward pass on random audio
6. Connects to Weights & Biases
7. Prints environment summary for documentation

**Estimated time:** 5-10 minutes the first time (mostly pip install).

## Step 1 — GPU check

Before anything else: are we on a GPU? If this cell shows "No GPU", go to **Runtime → Change runtime type → Hardware accelerator → GPU** and re-run.

**What you want to see:** Tesla T4, V100, or A100. With Colab Pro you'll usually get T4 (most common) or sometimes V100. A100 is rare on Pro.

In [ ]:
import subprocess

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('❌ No GPU detected. Go to Runtime → Change runtime type → GPU')

## Step 2 — Mount Google Drive

Colab gives you a temporary disk that wipes between sessions. We need persistent storage for the ~25 GB of ASVspoof data and our model checkpoints. Drive is the answer.

**You'll be prompted** to authorize access. Click the link, choose your Google account, copy the verification code, and paste it back. This happens once per session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Now create the folder structure on Drive. We do this once; it'll persist across sessions.

In [ ]:
import os

DRIVE_ROOT = '/content/drive/MyDrive/deepfake_audio'

subdirs = [
    'data/raw/asvspoof_2019',
    'data/raw/asvspoof_2021',
    'data/raw/wavefake',
    'data/processed',
    'checkpoints',
    'logs',
]

for sub in subdirs:
    path = os.path.join(DRIVE_ROOT, sub)
    os.makedirs(path, exist_ok=True)
    print(f'  ✅  {path}')

print('\n✅ Drive folder structure created.')

## Step 3 — Clone the GitHub repo

Pull our project code into Colab's local disk. We work out of `/content/deepfake-audio-detection/` for code, and read/write data via `/content/drive/MyDrive/deepfake_audio/`.

**⚠️ Replace `YOUR_USERNAME` below with your actual GitHub username before running.**

In [ ]:
# --- EDIT THIS LINE ---
REPO_URL = 'https://github.com/YOUR_USERNAME/deepfake-audio-detection.git'
# ----------------------

%cd /content
!if [ ! -d 'deepfake-audio-detection' ]; then git clone {REPO_URL}; else echo 'Repo already cloned'; fi
%cd /content/deepfake-audio-detection
!ls -la

## Step 4 — Install dependencies

We pin exact versions in `requirements.txt`. This protects us from "it worked yesterday" disasters when libraries silently update.

**Note:** after install, Colab may suggest restarting the runtime. Restart **once**, then continue from Step 5. Don't restart again later — that wipes the loaded model.

In [ ]:
!pip install -q -r requirements.txt
print('\n✅ Dependencies installed. If Colab shows a restart prompt, restart NOW and resume from Step 5.')

## Step 5 — Verify imports & versions

Confirm the libraries actually loaded with the versions we expect.

In [ ]:
import sys
import torch
import torchaudio
import transformers
import numpy as np
import pandas as pd
import librosa
import sklearn

print(f'Python:       {sys.version.split()[0]}')
print(f'PyTorch:      {torch.__version__}')
print(f'torchaudio:   {torchaudio.__version__}')
print(f'transformers: {transformers.__version__}')
print(f'numpy:        {np.__version__}')
print(f'pandas:       {pd.__version__}')
print(f'librosa:      {librosa.__version__}')
print(f'sklearn:      {sklearn.__version__}')

print(f'\nCUDA available:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA version:    {torch.version.cuda}')
    print(f'GPU name:        {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    print(f'GPU memory:      {props.total_memory / 1e9:.2f} GB')
    print(f'Compute capability: {props.major}.{props.minor}')

print('\n✅ All imports succeeded.')

## Step 6 — Load Wav2Vec 2.0

This is the core test. We load the pretrained Wav2Vec 2.0 Base model from HuggingFace. The first run downloads ~360 MB; subsequent runs use a cached copy.

**What's happening under the hood:** HuggingFace fetches the model weights and configuration, instantiates a PyTorch model, and we move it to the GPU.

In [ ]:
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor

MODEL_NAME = 'facebook/wav2vec2-base'

print(f'Loading {MODEL_NAME}...')
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)
model = Wav2Vec2Model.from_pretrained(MODEL_NAME)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'\n✅ Model loaded on {device}.')
print(f'   Total parameters: {n_params:,} (~{n_params/1e6:.1f}M)')
print(f'   Number of transformer layers: {model.config.num_hidden_layers}')
print(f'   Hidden size:                  {model.config.hidden_size}')

## Step 7 — Forward pass on random audio

Generate 4 seconds of random audio (just noise — we're testing plumbing, not real prediction). Push it through the model. Check the output shape makes sense.

**What you want to see:**
- Input shape: `(1, 64000)` — 1 batch × 4 seconds × 16,000 samples/sec
- Output shape: `(1, ~199, 768)` — 1 batch × ~199 time frames × 768 hidden dimensions
- Wav2Vec 2.0 downsamples by ~320, so 64000 / 320 ≈ 200 frames

In [ ]:
SAMPLE_RATE = 16000
DURATION_SEC = 4.0

# Random audio — Gaussian noise, just for shape testing
torch.manual_seed(42)
fake_audio = torch.randn(1, int(SAMPLE_RATE * DURATION_SEC)).to(device)
print(f'Input shape:  {tuple(fake_audio.shape)}')

with torch.no_grad():
    output = model(fake_audio)

hidden_states = output.last_hidden_state
print(f'Output shape: {tuple(hidden_states.shape)}')
print(f'   ↳ batch size: {hidden_states.shape[0]}')
print(f'   ↳ time frames: {hidden_states.shape[1]}  (one frame ≈ 20ms of audio)')
print(f'   ↳ feature dim: {hidden_states.shape[2]}  (Wav2Vec 2.0 Base uses 768)')

# Mean-pool over time to get one vector per clip — this is what our classifier head will use
pooled = hidden_states.mean(dim=1)
print(f'\nMean-pooled shape: {tuple(pooled.shape)}')
print(f'   ↳ This is what the classification head (added in Phase 3) will see.')

print('\n✅ Forward pass successful. The model is talking to the GPU correctly.')

## Step 8 — Connect to Weights & Biases

Wandb is our experiment tracking service. Without it, every Colab disconnect erases your training history. With it, every loss curve, every metric, every config — saved to the cloud and viewable in a browser.

**Before running this cell:**
1. Sign up at https://wandb.ai (use GitHub for one-click auth)
2. Go to https://wandb.ai/authorize and copy your API key
3. When prompted by the cell below, paste the key

After this works once, wandb caches your credentials in `/root/.netrc` for the session. You won't be prompted again unless you start fresh.

In [ ]:
import wandb

wandb.login()  # will prompt for your API key on first run

# Start a tiny test run to confirm everything connects end-to-end
run = wandb.init(
    project='deepfake-audio-detection',
    name='smoke-test',
    job_type='setup-test',
    config={
        'phase': 'phase-1-setup',
        'model': MODEL_NAME,
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'pytorch_version': torch.__version__,
    }
)

# Log a fake metric just to verify the pipe works
wandb.log({'smoke_test_metric': 1.0, 'environment_ok': True})

print(f'\n✅ Wandb run started.')
print(f'   View it at: {run.url}')

wandb.finish()
print('\n✅ Wandb test run finished. Visit the URL above to confirm it logged correctly.')

## Step 9 — Environment summary

Print the full environment fingerprint. **Copy this output into `docs/environment.md`** and commit it. Future-you (and your coauthor) will thank you.

In [ ]:
import platform
from datetime import datetime

print('='*70)
print(f'  ENVIRONMENT FINGERPRINT — {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print('='*70)
print(f'Platform:         Google Colab Pro')
print(f'OS:               {platform.platform()}')
print(f'Python:           {sys.version.split()[0]}')
print(f'PyTorch:          {torch.__version__}')
print(f'torchaudio:       {torchaudio.__version__}')
print(f'transformers:     {transformers.__version__}')
if torch.cuda.is_available():
    print(f'GPU:              {torch.cuda.get_device_name(0)}')
    print(f'CUDA:             {torch.version.cuda}')
    print(f'GPU memory:       {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
print(f'Drive root:       {DRIVE_ROOT}')
print(f'Wandb project:    deepfake-audio-detection')
print('='*70)

print('\n🎉 Phase 1 smoke test complete. You are ready for Phase 2 (data acquisition).')

## Phase 1 Checklist

Before moving to Phase 2, confirm all of these are done:

- [ ] All cells above ran without errors
- [ ] GPU detected (T4, V100, or A100)
- [ ] Drive mounted, `deepfake_audio/` folder structure created
- [ ] GitHub repo cloned into Colab
- [ ] Wav2Vec 2.0 loaded and forward pass succeeded
- [ ] Wandb run visible at the URL printed in Step 8
- [ ] Environment fingerprint copied into `docs/environment.md`
- [ ] Repo committed and pushed to GitHub with all Phase 1 files

**If anything failed:** scroll up, find the failing cell, read the error message, fix the issue, re-run. Don't proceed until every cell shows ✅.